# 앙상블 전용 (CPU 런타임)

GPU 없이 **저장된 확률 파일만으로** 앙상블 점수 확인 + 제출 파일 생성.
학습이 돌고 있는 동안 새 코랩 창(런타임 유형: CPU)에서 쓴다.

규칙 (EXPERIMENTS.md 핵심 결론)
- **가중치는 균등 평균만** — 가중치 탐색은 과적합 (홀드아웃 0.958 → LB 0.95084)
- **멤버 선택은 반분 검증** — H2000을 앞 1000 / 뒤 1000으로 나눠 **양쪽 모두 오를 때만** 추가
- 단독 점수가 높다고 좋은 멤버가 아니다. 다르게 틀리는 모델이 기여한다

셀 순서: 1 불러오기 → 2 고정 후보 비교 → 3 그리디 자동 선택 → 4 제출 파일

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, glob
R = "/content/drive/MyDrive/ssafy_ai"
S, J = f"{R}/sunghyun", f"{R}/jeongyeon"

# H2000 정답과 test id 순서 (둘 다 TEMPLATE 순서와 같다)
y   = pd.read_csv(f"{S}/hold_pred_ens_H2000_0.9580.csv")["answer"].str.strip().str.lower().values
ids = pd.read_csv(f"{S}/sub_ens_equal_7b_zs_q3_8b_ft_q3_32b_ft_ocr_H2000_0.9585.csv")["id"]

def lp(p):
    x = torch.load(p, weights_only=False); x = x.get("probs", x) if isinstance(x, dict) else x
    return np.asarray(x)
L   = np.array(list("abcd"))
acc = lambda P, s=slice(None): (L[P[s].argmax(1)] == y[s]).mean()
avg = lambda ks, D: sum(D[k] for k in ks) / len(ks)
H1, H2 = slice(0, 1000), slice(1000, 2000)

def tag(m, img, n, e=1): return f"{m}_choice_ce_shuf_img{img}_r16_n{n}_e{e}_H2000"
ENS = {   # 이름: (홀드아웃 H2000 확률, test 확률)
    "7b_zs":          (f"{S}/big2000_7b.pt",                    f"{S}/test_zeroshot_7b_img768.pt"),
    "7b_ft":          (f"{S}/valid_{tag('7b',1024,4714)}.pt",     f"{S}/test_{tag('7b',1024,4714)}_r0.pt"),
    "8b_ft768":       (f"{S}/valid_{tag('q3_8b',768,2000)}.pt",   f"{S}/test_{tag('q3_8b',768,2000)}_r0.pt"),
    "8b_ft768_n4714": (f"{S}/valid_{tag('q3_8b',768,4714)}.pt",   f"{S}/test_{tag('q3_8b',768,4714)}_r0.pt"),
    "8b_ft1024":      (f"{J}/valid_{tag('q3_8b',1024,4714)}.pt",  f"{J}/test_{tag('q3_8b',1024,4714)}_r0.pt"),
    "8b_ocr":         (f"{S}/valid_{tag('q3_8b',1024,4714)}_ocr.pt", f"{S}/test_{tag('q3_8b',1024,4714)}_ocr_r0.pt"),
    "32b_768":        (f"{S}/valid_{tag('q3_32b',768,4714)}.pt",  f"{S}/test_{tag('q3_32b',768,4714)}_r0.pt"),
    "32b_1024":       (f"{S}/valid_{tag('q3_32b',1024,4714)}.pt", f"{S}/test_{tag('q3_32b',1024,4714)}_r0.pt"),
    "4b_ft":          (f"{S}/valid_{tag('q3_4b',1024,4714)}.pt",  f"{S}/test_{tag('q3_4b',1024,4714)}_r0.pt"),
}
V, T = {}, {}
for k, (v, t) in ENS.items():
    V[k] = lp(v)
    try:
        T[k] = lp(t); assert T[k].shape == (len(ids), 4)
    except Exception:
        T.pop(k, None); print(f"  ({k}: test 확률 없음/미완성 → 제출에는 못 씀)")
    print(f"{k:16s} {acc(V[k]):.4f}")

In [ ]:
# 고정 후보 비교 — 결과를 보고 고르지 말고, 이유를 먼저 정해서 적는다
CANDS = {
    "E7 (LB 0.95978)": ["7b_zs", "8b_ft768", "8b_ft1024", "32b_768"],
    "E8 (LB 0.96336)": ["7b_zs", "8b_ft768", "32b_768", "32b_1024"],
    "E9 그리디":        ["32b_768", "8b_ft1024", "32b_1024"],
}
for name, ks in CANDS.items():
    P = avg(ks, V)
    print(f"{name:16s} 전체 {acc(P):.4f} | 앞 {acc(P,H1):.4f} | 뒤 {acc(P,H2):.4f}   {' + '.join(ks)}")
    for d in ks:
        r = [k for k in ks if k != d]
        print(f"   - {d:16s} 빼면 {acc(avg(r, V)):.4f}")

In [ ]:
# 그리디: 앞/뒤 1000건이 둘 다 나빠지지 않고 합이 오를 때만 멤버 추가
cur, base = [], (0, 0)
while True:
    best = None
    for k in V:
        if k in cur: continue
        P = avg(cur + [k], V); a, b = acc(P, H1), acc(P, H2)
        if not cur or (a >= base[0] and b >= base[1] and a + b > sum(base)):
            if best is None or a + b > best[1] + best[2]: best = (k, a, b)
    if best is None: break
    cur.append(best[0]); base = best[1:]
    print(f"+ {best[0]:16s} → 전체 {acc(avg(cur, V)):.4f} | 앞 {best[1]:.4f} | 뒤 {best[2]:.4f}")
print("\n그리디 선택:", cur)

In [ ]:
# 제출 파일 — 쓸 조합을 직접 지정
PICK = CANDS["E8 (LB 0.96336)"]          # 또는 cur (그리디 결과)
assert all(k in T for k in PICK), [k for k in PICK if k not in T]
s   = acc(avg(PICK, V))
sub = f"sub_ens_equal_{'_'.join(PICK)}_H2000_{s:.4f}.csv"
out = pd.DataFrame({"id": ids, "answer": L[avg(PICK, T).argmax(1)]})
out.to_csv(f"{S}/{sub}", index=False); out.to_csv(f"/content/{sub}", index=False)
print(sub, "| 행", len(out), "| 분포", out["answer"].value_counts(normalize=True).round(3).to_dict())